## 1. Importation et Nettoyage des Données
Nous configurons l'environnement, chargeons ou simulons le jeu de données historique et gérons les anomalies/valeurs manquantes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Simulation robuste du jeu de données historique
np.random.seed(42)
n_records = 2000

dates = pd.date_range(start='1950-01-01', end='2023-12-31', periods=n_records)
operators = np.random.choice(['Aeroflot', 'Military', 'Air France', 'Lufthansa', 'United Airlines', 'Private'], n_records)
aboard = np.random.randint(5, 300, n_records)
fatalities = np.array([np.random.randint(0, a + 1) for a in aboard])

df = pd.DataFrame({
    'Date': dates,
    'Operator': operators,
    'Aboard': aboard,
    'Fatalities': fatalities
})

# Ajout artificiel de valeurs manquantes
df.loc[df.sample(frac=0.02).index, 'Operator'] = np.nan

# Nettoyage
df['Operator'] = df['Operator'].fillna('Unknown')
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Decade'] = (df['Year'] // 10) * 10
df['Survival_Rate'] = (df['Aboard'] - df['Fatalities']) / df['Aboard']

print("Données nettoyées avec succès. Aperçu :")
print(df.head())

## 2. Analyse Exploratoire des Données (EDA)
Extraction des grandes tendances et des statistiques globales du trafic accidenté.

In [ ]:
total_crashes = len(df)
total_aboard = df['Aboard'].sum()
total_fatalities = df['Fatalities'].sum()
global_survival_rate = (total_aboard - total_fatalities) / total_aboard

print(f"Nombre total d'accidents : {total_crashes}")
print(f"Total de passagers impliqués : {total_aboard}")
print(f"Total des décès : {total_fatalities}")
print(f"Taux de survie global : {global_survival_rate:.2%}")

## 3. Analyses Statistiques Inférentielles (SciPy)
Calcul des indices de distribution et exécution d'un test T de Student indépendant pour comparer les décès des années 1970 à ceux des années 2010.

In [ ]:
mean_fat = np.mean(df['Fatalities'])
median_fat = np.median(df['Fatalities'])
std_fat = stats.tstd(df['Fatalities'])

print(f"Moyenne des décès par accident : {mean_fat:.2f}")
print(f"Médiane des décès par accident : {median_fat:.2f}")
print(f"Écart-type des décès par accident : {std_fat:.2f}\n")

fatalities_1970 = df[df['Decade'] == 1970]['Fatalities']
fatalities_2010 = df[df['Decade'] == 2010]['Fatalities']

stat_t, p_val = stats.ttest_ind(fatalities_1970, fatalities_2010, equal_var=False)
print(f"Statistique T du test : {stat_t:.4f}")
print(f"P-valeur du test : {p_val:.4e}")

## 4. Visualisations Graphiques
Génération des courbes d'évolution temporelle, de l'histogramme des mortalités et des boxplots comparatifs.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

# Évolution temporelle
df_yearly = df.groupby('Year').size().reset_index(name='Count')
sns.lineplot(data=df_yearly, x='Year', y='Count', ax=axes[0], color='darkred', linewidth=2.5)
axes[0].set_title("Évolution du nombre annuel d'accidents d'avion")

# Histogramme des décès
sns.histplot(df['Fatalities'], bins=30, kde=True, ax=axes[1], color='teal')
axes[1].set_title("Distribution du nombre de décès par accident")

# Boxplot par décennie
sns.boxplot(data=df, x='Decade', y='Survival_Rate', ax=axes[2], palette='Blues')
axes[2].set_title("Distribution du taux de survie aux accidents par décennie")

plt.tight_layout()
plt.show()